# TsPlots — 自定义绘图工具包

本 Notebook 演示 `TsPlots` 包的完整用法，包括：

1. **plot_series** — 时间序列折线图
2. **plot_scatter** — 散点图
3. **plot_acf / plot_pacf** — 自相关 / 偏自相关图
4. **样式定制** — 调色板、字体、图例、阴影、参考线

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Ts.TsPlots import plot_series, plot_scatter, plot_acf, plot_pacf
from Ts.TsPlots.style import DEFAULT_PALETTE, style_axes, apply_fonts

# 准备演示数据
np.random.seed(42)
t = np.arange(100)
df = pd.DataFrame({
    "GDP": np.cumsum(np.random.randn(100)) + 100,
    "Consumption": np.cumsum(np.random.randn(100) * 0.8) + 80,
    "Investment": np.cumsum(np.random.randn(100) * 1.5) + 30,
    "year": 1920 + t,
}).set_index("year")

print(df.head())

---
## 1. plot_series — 时间序列折线图

### 1.1 基础用法 — 单序列

In [ ]:
fig, ax = plot_series(df, y="GDP", title="GDP Time Series", ytitle="Value")
plt.show()

### 1.2 多序列 — DataFrame 多列自动分色

In [ ]:
fig, ax = plot_series(
    df[["GDP", "Consumption", "Investment"]],
    title="Macroeconomic Indicators",
    ytitle="Index",
    markersize=0,
    linewidth=2,
)
plt.show()

### 1.3 dict 输入 — 自定义名称

In [ ]:
data_dict = {"GDP": df["GDP"].values, "消费": df["Consumption"].values}
fig, ax = plot_series(
    data_dict, x=t,
    title="中英文混排标签", ytitle="指数",
    linewidth=2, markersize=3,
    unit="指数",
)
plt.show()

### 1.4 阴影区间 (shade)

In [ ]:
fig, ax = plot_series(
    df["GDP"],
    title="GDP with Recession Shading",
    ytitle="Index",
    markersize=0, linewidth=2,
    shade=[(30, 40), (70, 80)],  # 阴影区间 (x 索引)
    shade_color="#ff9999", shade_alpha=0.3,
)
plt.show()

### 1.5 竖线标注 (vlines)

In [ ]:
fig, ax = plot_series(
    df["GDP"],
    title="GDP with Event Lines",
    ytitle="Index",
    markersize=0, linewidth=2,
    vlines=[25, 75],
    vline_color="#D55E00", vline_linestyle="--",
)
plt.show()

### 1.6 数值标注 (show_values)

In [ ]:
short_series = pd.Series([10.2, 12.5, 9.8, 15.3, 11.7], name="Sales")
fig, ax = plot_series(
    short_series,
    title="Sales with Value Labels",
    ytitle="Million USD",
    show_values=True, value_decimals=1,
    markersize=10,
)
plt.show()

### 1.7 网格 + 自定义颜色 + 标题位置底部

In [ ]:
fig, ax = plot_series(
    {"GDP": df["GDP"], "C": df["Consumption"]},
    title="底部标题示例", title_position="bottom",
    ytitle="指数", grid=True,
    colors=["#0072B2", "#D55E00"],
    markersize=0, linewidth=2,
)
plt.show()

---
## 2. plot_scatter — 散点图

### 2.1 基础散点 — DataFrame 列

In [ ]:
fig, ax = plot_scatter(
    df, x="GDP", y="Consumption",
    title="Consumption vs GDP",
    show_legend=False,
)
plt.show()

### 2.2 带 OLS 趋势线 (fit_line=True)

In [ ]:
fig, ax = plot_scatter(
    df, x="GDP", y="Consumption",
    title="Consumption vs GDP (with OLS fit)",
    fit_line=True, fit_linewidth=2.5, fit_linestyle="--",
    show_legend=False,
)
plt.show()

### 2.3 分组散点 — group 参数

In [ ]:
# 构造分组数据
group_df = pd.DataFrame({
    "x": np.random.randn(60),
    "y": np.random.randn(60),
    "category": ["A"] * 20 + ["B"] * 20 + ["C"] * 20,
})

fig, ax = plot_scatter(
    group_df, x="x", y="y", group="category",
    title="Grouped Scatter Plot",
)
plt.show()

### 2.4 竖线 / 横线 + 数值标注

In [ ]:
fig, ax = plot_scatter(
    df, x="GDP", y="Consumption",
    title="Scatter with Reference Lines",
    vlines=[100], hlines=[100],
    vline_color="#D55E00", hline_color="#0072B2",
    show_legend=False,
)
plt.show()

### 2.5 等比例坐标轴 (equal_aspect)

In [ ]:
x_returns = np.random.randn(100)
y_returns = np.random.randn(100)
fig, ax = plot_scatter(
    x=x_returns, y=y_returns,
    title="Asset Returns (equal aspect)",
    equal_aspect=True, alpha=0.5, show_legend=False,
)
plt.show()


---
## 3. plot_acf / plot_pacf — 自相关图

### 3.1 ACF 图 — AR(1) 过程

In [ ]:
# 生成 AR(1) φ=0.7
from statsmodels.tsa.arima_process import ArmaProcess
ar = np.array([1, -0.7])
ma = np.array([1])
ar1_data = ArmaProcess(ar, ma).generate_sample(200)

fig, ax = plot_acf(ar1_data, nlags=30, title="ACF of AR(1) with φ=0.7")
plt.show()

### 3.2 PACF 图

In [ ]:
fig, ax = plot_pacf(ar1_data, nlags=30, title="PACF of AR(1) with φ=0.7")
plt.show()

### 3.3 ACF — 去掉 lag 0

In [ ]:
fig, ax = plot_acf(
    ar1_data, nlags=30, zero_lag=False,
    title="ACF (without lag 0)",
)
plt.show()

### 3.4 ACF — 99% 置信带

In [ ]:
fig, ax = plot_acf(
    ar1_data, nlags=30, alpha=0.01,
    title="ACF with 99% Confidence Band",
    bar_color="#D55E00",
)
plt.show()

---
## 4. 样式系统

`TsPlots.style` 模块提供全局样式控制。

### 4.1 调色板 (Okabe-Ito 8色)

In [ ]:
from Ts.TsPlots.style import DEFAULT_PALETTE

fig, ax = plt.subplots(figsize=(8, 2))
for i, color in enumerate(DEFAULT_PALETTE):
    ax.add_patch(plt.Rectangle((i, 0), 0.8, 1, color=color))
    ax.text(i + 0.4, -0.4, f"{i}", ha="center", fontsize=9)
ax.set_xlim(-0.5, len(DEFAULT_PALETTE) + 0.5)
ax.set_ylim(-1, 2)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"DEFAULT_PALETTE ({len(DEFAULT_PALETTE)} colors, Okabe-Ito)")
plt.show()

### 4.2 字体系统 — Times New Roman + 仿宋

In [ ]:
from Ts.TsPlots.style import apply_fonts, LATIN_FONT, CHINESE_FONT_CANDIDATES

print(f"Latin font: {LATIN_FONT}")
print(f"CJK candidates: {CHINESE_FONT_CANDIDATES}")

fig, ax = plt.subplots(figsize=(6, 2))
ax.text(0.5, 0.5, "English 中文 日本語 한글", ha="center", va="center", fontsize=16, transform=ax.transAxes)
ax.set_title("中英文混排测试 (Times New Roman + 仿宋)")
style_axes(ax)
plt.show()

---
## 小结

| 函数 | 用途 | 关键参数 |
|------|------|----------|
| `plot_series()` | 时间序列折线图 | `data`, `x`, `y`, `shade`, `vlines`, `show_values`, `title_position` |
| `plot_scatter()` | 散点图 | `data`, `x`, `y`, `group`, `fit_line`, `hlines`, `vlines` |
| `plot_acf()` | 自相关图 | `data`, `nlags`, `alpha`, `zero_lag`, `bartlett_confint` |
| `plot_pacf()` | 偏自相关图 | `data`, `nlags`, `alpha` |

**样式常量** (from `TsPlots.style`)：
- `DEFAULT_PALETTE` — 8色调色板 (Okabe-Ito)
- `LATIN_FONT` / `CHINESE_FONT_CANDIDATES` — 字体配置
- `style_axes(ax)` — 统一坐标轴样式
- `apply_fonts()` — 应用字体配置